In [1]:
from pyspark.sql import SparkSession
spark=SparkSession.builder\
    .appName("MySparkSessionDemo")\
    .master("local[*]")\
    .config("spark.sql.shuffle.partitions","4")\
    .getOrCreate()

print("SparkSession created")
print("AppName",spark.sparkContext.appName)
print("master",spark.sparkContext.master)
print("Spark Version",spark.version)

SparkSession created
AppName MySparkSessionDemo
master local[*]
Spark Version 3.5.8


In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("WindowFunctions").getOrCreate()

In [3]:
data = [
    (1, "Alice", "HR", "Mumbai", 50000),
    (2, "Bob", "IT", "Bangalore", 70000),
    (3, "Charlie", "IT", "Mumbai", 80000),
    (4, "David", "HR", "Delhi", 60000),
    (5, "Eve", "Sales", "Mumbai", 75000),
    (6, "Frank", "Sales", "Delhi", 72000),
    (7, "Grace", "IT", "Bangalore", 90000),
    (8, "Helen", "HR", "Mumbai", 65000)
]

columns = ["id", "name", "department", "city", "salary"]

df = spark.createDataFrame(data, columns)
df.show()

+---+-------+----------+---------+------+
| id|   name|department|     city|salary|
+---+-------+----------+---------+------+
|  1|  Alice|        HR|   Mumbai| 50000|
|  2|    Bob|        IT|Bangalore| 70000|
|  3|Charlie|        IT|   Mumbai| 80000|
|  4|  David|        HR|    Delhi| 60000|
|  5|    Eve|     Sales|   Mumbai| 75000|
|  6|  Frank|     Sales|    Delhi| 72000|
|  7|  Grace|        IT|Bangalore| 90000|
|  8|  Helen|        HR|   Mumbai| 65000|
+---+-------+----------+---------+------+



In [ ]:
#in group by number of rows reduced in window rows retained and calculation performed
w_spec=Window.partitionBy("department")
df.withColumn("average_sal",F.avg("salary").over(w_spec)).show()

+---+-------+----------+---------+------+------------------+
| id|   name|department|     city|salary|       average_sal|
+---+-------+----------+---------+------+------------------+
|  1|  Alice|        HR|   Mumbai| 50000|58333.333333333336|
|  4|  David|        HR|    Delhi| 60000|58333.333333333336|
|  8|  Helen|        HR|   Mumbai| 65000|58333.333333333336|
|  2|    Bob|        IT|Bangalore| 70000|           80000.0|
|  3|Charlie|        IT|   Mumbai| 80000|           80000.0|
|  7|  Grace|        IT|Bangalore| 90000|           80000.0|
|  5|    Eve|     Sales|   Mumbai| 75000|           73500.0|
|  6|  Frank|     Sales|    Delhi| 72000|           73500.0|
+---+-------+----------+---------+------+------------------+



In [6]:
df.groupBy("department").agg(
    F.avg("salary")
).show()

+----------+------------------+
|department|       avg(salary)|
+----------+------------------+
|        HR|58333.333333333336|
|        IT|           80000.0|
|     Sales|           73500.0|
+----------+------------------+



In [7]:
w_spec=Window.partitionBy("department")
df.agg(
    F.avg("salary").over(w_spec)).show()
    #we cant use agg in windows so using it withcolumn


AnalysisException: [MISSING_GROUP_BY] The query does not include a GROUP BY clause. Add GROUP BY or turn it into the window functions using OVER clauses.;
Project [avg(salary) OVER (PARTITION BY department unspecifiedframe$())#66]
+- Project [salary#4L, department#2, avg(salary) OVER (PARTITION BY department unspecifiedframe$())#66, avg(salary) OVER (PARTITION BY department unspecifiedframe$())#66]
   +- Window [avg(salary#4L) windowspecdefinition(department#2, specifiedwindowframe(RowFrame, unboundedpreceding$(), unboundedfollowing$())) AS avg(salary) OVER (PARTITION BY department unspecifiedframe$())#66], [department#2]
      +- Aggregate [salary#4L, department#2]
         +- LogicalRDD [id#0L, name#1, department#2, city#3, salary#4L], false


In [8]:
window_spec = Window.partitionBy("department").orderBy(F.col("salary").desc())

df.withColumn(
    "ordered_salary",
    F.row_number().over(window_spec)
).show()

+---+-------+----------+---------+------+--------------+
| id|   name|department|     city|salary|ordered_salary|
+---+-------+----------+---------+------+--------------+
|  8|  Helen|        HR|   Mumbai| 65000|             1|
|  4|  David|        HR|    Delhi| 60000|             2|
|  1|  Alice|        HR|   Mumbai| 50000|             3|
|  7|  Grace|        IT|Bangalore| 90000|             1|
|  3|Charlie|        IT|   Mumbai| 80000|             2|
|  2|    Bob|        IT|Bangalore| 70000|             3|
|  5|    Eve|     Sales|   Mumbai| 75000|             1|
|  6|  Frank|     Sales|    Delhi| 72000|             2|
+---+-------+----------+---------+------+--------------+



In [16]:
window_spec = Window.partitionBy("department").orderBy(F.col("salary").desc())
df.select("*",F.row_number().over(window_spec)).show()


+---+-------+----------+---------+------+----------------------------------------------------------------------------------------------------------------------------+
| id|   name|department|     city|salary|row_number() OVER (PARTITION BY department ORDER BY salary DESC NULLS LAST ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)|
+---+-------+----------+---------+------+----------------------------------------------------------------------------------------------------------------------------+
|  8|  Helen|        HR|   Mumbai| 65000|                                                                                                                           1|
|  4|  David|        HR|    Delhi| 60000|                                                                                                                           2|
|  1|  Alice|        HR|   Mumbai| 50000|                                                                                                                           3

Important Rule

.over(window_spec) works only with window functions like:

row_number()

rank()

dense_rank()

sum()

avg()

lag()

lead()

In [17]:
df.withColumn(
    "rank",
    F.rank().over(window_spec)
).show()

+---+-------+----------+---------+------+----+
| id|   name|department|     city|salary|rank|
+---+-------+----------+---------+------+----+
|  8|  Helen|        HR|   Mumbai| 65000|   1|
|  4|  David|        HR|    Delhi| 60000|   2|
|  1|  Alice|        HR|   Mumbai| 50000|   3|
|  7|  Grace|        IT|Bangalore| 90000|   1|
|  3|Charlie|        IT|   Mumbai| 80000|   2|
|  2|    Bob|        IT|Bangalore| 70000|   3|
|  5|    Eve|     Sales|   Mumbai| 75000|   1|
|  6|  Frank|     Sales|    Delhi| 72000|   2|
+---+-------+----------+---------+------+----+



In [18]:
df.withColumn(
    "dense_rank",
    F.dense_rank().over(window_spec)
).show()

+---+-------+----------+---------+------+----------+
| id|   name|department|     city|salary|dense_rank|
+---+-------+----------+---------+------+----------+
|  8|  Helen|        HR|   Mumbai| 65000|         1|
|  4|  David|        HR|    Delhi| 60000|         2|
|  1|  Alice|        HR|   Mumbai| 50000|         3|
|  7|  Grace|        IT|Bangalore| 90000|         1|
|  3|Charlie|        IT|   Mumbai| 80000|         2|
|  2|    Bob|        IT|Bangalore| 70000|         3|
|  5|    Eve|     Sales|   Mumbai| 75000|         1|
|  6|  Frank|     Sales|    Delhi| 72000|         2|
+---+-------+----------+---------+------+----------+



In [19]:
from pyspark.sql.window import Window

data = [
    (101, "Alice",   "HR",    "Mumbai",    "2024-01-10", 50000),
    (102, "Bob",     "IT",    "Bangalore", "2024-01-12", 70000),
    (103, "Charlie", "IT",    "Mumbai",    "2024-01-15", 80000),
    (104, "David",   "HR",    "Delhi",     "2024-01-18", 60000),
    (105, "Eve",     "Sales", "Mumbai",    "2024-01-20", 75000),
    (106, "Frank",   "Sales", "Delhi",     "2024-01-22", 72000),
    (107, "Grace",   "IT",    "Bangalore", "2024-01-25", 90000),
    (108, "Helen",   "HR",    "Mumbai",    "2024-01-28", 65000),
    (109, "Ivy",     "IT",    "Delhi",     "2024-02-01", 70000),
    (110, "Jack",    "Sales", "Bangalore", "2024-02-03", 68000),
    (111, "Kevin",   "HR",    "Delhi",     "2024-02-05", 60000),
    (112, "Lara",    "IT",    "Mumbai",    "2024-02-08", 85000),
    (113, "Mona",    "Sales", "Mumbai",    "2024-02-10", 75000),
    (114, "Nina",    "HR",    "Bangalore", "2024-02-12", 58000),
    (115, "Oscar",   "IT",    "Delhi",     "2024-02-15", 95000),
    (116, "Paul",    "Sales", "Delhi",     "2024-02-18", 72000),
    (117, "Quinn",   "HR",    "Mumbai",    "2024-02-20", 67000),
    (118, "Rose",    "IT",    "Bangalore", "2024-02-22", 88000),
    (119, "Sam",     "Sales", "Mumbai",    "2024-02-25", 79000),
    (120, "Tina",    "HR",    "Delhi",     "2024-02-28", 61000)
]

columns = ["emp_id", "name", "department", "city", "joining_date", "salary"]

df = spark.createDataFrame(data, columns)

df = df.withColumn("joining_date", F.to_date("joining_date", "yyyy-MM-dd"))

df.show(truncate=False)
df.printSchema()

+------+-------+----------+---------+------------+------+
|emp_id|name   |department|city     |joining_date|salary|
+------+-------+----------+---------+------------+------+
|101   |Alice  |HR        |Mumbai   |2024-01-10  |50000 |
|102   |Bob    |IT        |Bangalore|2024-01-12  |70000 |
|103   |Charlie|IT        |Mumbai   |2024-01-15  |80000 |
|104   |David  |HR        |Delhi    |2024-01-18  |60000 |
|105   |Eve    |Sales     |Mumbai   |2024-01-20  |75000 |
|106   |Frank  |Sales     |Delhi    |2024-01-22  |72000 |
|107   |Grace  |IT        |Bangalore|2024-01-25  |90000 |
|108   |Helen  |HR        |Mumbai   |2024-01-28  |65000 |
|109   |Ivy    |IT        |Delhi    |2024-02-01  |70000 |
|110   |Jack   |Sales     |Bangalore|2024-02-03  |68000 |
|111   |Kevin  |HR        |Delhi    |2024-02-05  |60000 |
|112   |Lara   |IT        |Mumbai   |2024-02-08  |85000 |
|113   |Mona   |Sales     |Mumbai   |2024-02-10  |75000 |
|114   |Nina   |HR        |Bangalore|2024-02-12  |58000 |
|115   |Oscar 

In [20]:
df.show()

+------+-------+----------+---------+------------+------+
|emp_id|   name|department|     city|joining_date|salary|
+------+-------+----------+---------+------------+------+
|   101|  Alice|        HR|   Mumbai|  2024-01-10| 50000|
|   102|    Bob|        IT|Bangalore|  2024-01-12| 70000|
|   103|Charlie|        IT|   Mumbai|  2024-01-15| 80000|
|   104|  David|        HR|    Delhi|  2024-01-18| 60000|
|   105|    Eve|     Sales|   Mumbai|  2024-01-20| 75000|
|   106|  Frank|     Sales|    Delhi|  2024-01-22| 72000|
|   107|  Grace|        IT|Bangalore|  2024-01-25| 90000|
|   108|  Helen|        HR|   Mumbai|  2024-01-28| 65000|
|   109|    Ivy|        IT|    Delhi|  2024-02-01| 70000|
|   110|   Jack|     Sales|Bangalore|  2024-02-03| 68000|
|   111|  Kevin|        HR|    Delhi|  2024-02-05| 60000|
|   112|   Lara|        IT|   Mumbai|  2024-02-08| 85000|
|   113|   Mona|     Sales|   Mumbai|  2024-02-10| 75000|
|   114|   Nina|        HR|Bangalore|  2024-02-12| 58000|
|   115|  Osca

In [25]:
w_spec=Window.partitionBy("department").orderBy(F.col("salary").desc())
df.withColumn("rank",
              F.row_number().over(w_spec)
).filter(F.col("rank")==1).show()

+------+-----+----------+------+------------+------+----+
|emp_id| name|department|  city|joining_date|salary|rank|
+------+-----+----------+------+------------+------+----+
|   117|Quinn|        HR|Mumbai|  2024-02-20| 67000|   1|
|   115|Oscar|        IT| Delhi|  2024-02-15| 95000|   1|
|   119|  Sam|     Sales|Mumbai|  2024-02-25| 79000|   1|
+------+-----+----------+------+------------+------+----+



In [27]:
total_df=df.groupBy("department","city").agg(
    F.sum("salary").alias("total_salary")
)
w_spec=Window.partitionBy("department").orderBy(F.col("total_salary").desc())
rank=total_df.withColumn(
    "rank",F.row_number().over(w_spec)
).filter(F.col("rank")<=2).show()

+----------+---------+------------+----+
|department|     city|total_salary|rank|
+----------+---------+------------+----+
|        HR|   Mumbai|      182000|   1|
|        HR|    Delhi|      181000|   2|
|        IT|Bangalore|      248000|   1|
|        IT|   Mumbai|      165000|   2|
|     Sales|   Mumbai|      229000|   1|
|     Sales|    Delhi|      144000|   2|
+----------+---------+------------+----+



In [33]:
df_dup=df.union(df.filter("emp_id IN (101,102,103)"))
df_dup.show(25,truncate=False)

+------+-------+----------+---------+------------+------+
|emp_id|name   |department|city     |joining_date|salary|
+------+-------+----------+---------+------------+------+
|101   |Alice  |HR        |Mumbai   |2024-01-10  |50000 |
|102   |Bob    |IT        |Bangalore|2024-01-12  |70000 |
|103   |Charlie|IT        |Mumbai   |2024-01-15  |80000 |
|104   |David  |HR        |Delhi    |2024-01-18  |60000 |
|105   |Eve    |Sales     |Mumbai   |2024-01-20  |75000 |
|106   |Frank  |Sales     |Delhi    |2024-01-22  |72000 |
|107   |Grace  |IT        |Bangalore|2024-01-25  |90000 |
|108   |Helen  |HR        |Mumbai   |2024-01-28  |65000 |
|109   |Ivy    |IT        |Delhi    |2024-02-01  |70000 |
|110   |Jack   |Sales     |Bangalore|2024-02-03  |68000 |
|111   |Kevin  |HR        |Delhi    |2024-02-05  |60000 |
|112   |Lara   |IT        |Mumbai   |2024-02-08  |85000 |
|113   |Mona   |Sales     |Mumbai   |2024-02-10  |75000 |
|114   |Nina   |HR        |Bangalore|2024-02-12  |58000 |
|115   |Oscar 

In [31]:
df_dup.count()

20

In [43]:
window_spec=Window.partitionBy("department").orderBy(F.col("salary").desc())

df_ranked=df_dup.withColumn(
    "row_num",
    F.row_number().over(window_spec)
)
df_ranked.show(25,truncate=False)

+------+-------+----------+---------+------------+------+-------+
|emp_id|name   |department|city     |joining_date|salary|row_num|
+------+-------+----------+---------+------------+------+-------+
|117   |Quinn  |HR        |Mumbai   |2024-02-20  |67000 |1      |
|108   |Helen  |HR        |Mumbai   |2024-01-28  |65000 |2      |
|120   |Tina   |HR        |Delhi    |2024-02-28  |61000 |3      |
|104   |David  |HR        |Delhi    |2024-01-18  |60000 |4      |
|111   |Kevin  |HR        |Delhi    |2024-02-05  |60000 |5      |
|114   |Nina   |HR        |Bangalore|2024-02-12  |58000 |6      |
|101   |Alice  |HR        |Mumbai   |2024-01-10  |50000 |7      |
|101   |Alice  |HR        |Mumbai   |2024-01-10  |50000 |8      |
|115   |Oscar  |IT        |Delhi    |2024-02-15  |95000 |1      |
|107   |Grace  |IT        |Bangalore|2024-01-25  |90000 |2      |
|118   |Rose   |IT        |Bangalore|2024-02-22  |88000 |3      |
|112   |Lara   |IT        |Mumbai   |2024-02-08  |85000 |4      |
|103   |Ch

In [44]:
window_salary=Window.partitionBy("department","salary")
df_final=df_ranked.withColumn(
    "salary_count",
    F.count("*").over(window_salary)
)

result=df_final.filter(
    F.col("salary_count")>1
)
result.show()

+------+-------+----------+---------+------------+------+-------+------------+
|emp_id|   name|department|     city|joining_date|salary|row_num|salary_count|
+------+-------+----------+---------+------------+------+-------+------------+
|   101|  Alice|        HR|   Mumbai|  2024-01-10| 50000|      7|           2|
|   101|  Alice|        HR|   Mumbai|  2024-01-10| 50000|      8|           2|
|   104|  David|        HR|    Delhi|  2024-01-18| 60000|      4|           2|
|   111|  Kevin|        HR|    Delhi|  2024-02-05| 60000|      5|           2|
|   102|    Bob|        IT|Bangalore|  2024-01-12| 70000|      7|           3|
|   109|    Ivy|        IT|    Delhi|  2024-02-01| 70000|      8|           3|
|   102|    Bob|        IT|Bangalore|  2024-01-12| 70000|      9|           3|
|   103|Charlie|        IT|   Mumbai|  2024-01-15| 80000|      5|           2|
|   103|Charlie|        IT|   Mumbai|  2024-01-15| 80000|      6|           2|
|   106|  Frank|     Sales|    Delhi|  2024-01-22| 7